In [1]:
import pandas as pd # importando biblioteca essencial

In [2]:
df = pd.read_excel('../data/data.xlsx') # lendo o arquivo excel e armazenando em um DataFrame

In [3]:
df.info() # exibindo informações sobre o DataFrame, como número de linhas, colunas e tipos de dados

<class 'pandas.DataFrame'>
RangeIndex: 452 entries, 0 to 451
Data columns (total 20 columns):
 #   Column                                              Non-Null Count  Dtype  
---  ------                                              --------------  -----  
 0   Município de vacinação                              444 non-null    str    
 1   Idade Evento                                        444 non-null    float64
 2   Data da Notificação                                 444 non-null    object 
 3   Raça/Cor                                            444 non-null    object 
 4   Sexo                                                444 non-null    str    
 5   Imunobiológico (vacina)                             444 non-null    str    
 6   Dose                                                443 non-null    str    
 7   Gestante                                            444 non-null    object 
 8   Mês de gestação                                     444 non-null    float64
 9   Mulher ama

# Tratamento da coluna data

A coluna data não segue uma padronização clara. Enquanto algumas linhas apresentam apenas a informação do ano, outras apresentam a data completa.

É necessário tratar e padronizar a coluna de data.

In [4]:
# Convertendo a coluna 'Data da Notificação' para string e removendo espaços em branco
df['Data da Notificação'] = df['Data da Notificação'].astype(str).str.strip()

In [5]:
# Extraindo o ano da notificação e criando uma nova coluna 'Ano_Notificacao'
df['Ano_Notificacao'] = df['Data da Notificação'].str[:4]

In [6]:
# Convertendo de volta para o formato correto
df['Data_Completa_Notificacao'] = pd.to_datetime(df['Data da Notificação'], errors='coerce')

# Tratando múltiplas informações nas colunas vacina e dose

As colunas vacina e dose apresentam mais de uma informação por entrada.
É de extrema importância dividir esses informações em colunas adjacentes, visando facilitar a busca, leitura e processamento dos dados nessas colunas.

In [7]:
# separando as vacinas pelo separador / e criando novas colunas para cada vacina
vacinas_separadas = df['Imunobiológico (vacina)'].str.split('/', expand=True).add_prefix('Vacina_')

In [8]:
# separando as doses pelo separador / e criando novas colunas para cada dose
doses_separadas = df['Dose'].str.split('/', expand=True).add_prefix('Dose_')

In [9]:
for col in vacinas_separadas.columns:
    # Remove espaços em branco nas pontas
    vacinas_separadas[col] = vacinas_separadas[col].str.strip()
    # Remove o padrão de numeração inicial (ex: "1: ") se ele existir
    vacinas_separadas[col] = vacinas_separadas[col].str.replace(r'^\d+:\s*', '', regex=True)

for col in doses_separadas.columns:
    doses_separadas[col] = doses_separadas[col].str.strip()
    doses_separadas[col] = doses_separadas[col].str.replace(r'^\d+:\s*', '', regex=True)

In [10]:
df = pd.concat([df, vacinas_separadas, doses_separadas], axis=1) # juntando as novas colunas de vacinas e doses ao DataFrame original

# Limpeza e Padronização dos Eventos e Desfechos

Vamos aplicar uma limpeza em lote nessas colunas de uma vez só, usando expressões regulares (regex) para remover os prefixos numéricos e os espaços em branco que sobram nas pontas.

In [ ]:
colunas_para_limpar = [
    'Tipo de Evento', 
    'Reação / evento adverso', 
    'Classificação de gravidade', 
    'Desfecho (evolução do caso)'
]

for col in colunas_para_limpar:
    # Garantir que a coluna seja tratada como texto (string)
    df[col] = df[col].astype(str)
    
    # Remover o padrão numérico do início 
    df[col] = df[col].str.replace(r'^\d+:\s*', '', regex=True)
    
    # Remover espaços em branco inúteis no início e no final do texto
    df[col] = df[col].str.strip()
    
    # Substituir textos que viraram "nan" (por estarem vazios antes) por um valor nulo real (None)
    df[col] = df[col].replace(['nan', 'NaN', '0'], None)

# 1. Separamos o texto de 'Tipo de Evento' onde tem '/' gerando uma lista
df['Tipo de Evento'] = df['Tipo de Evento'].str.split('/')

# 2. O explode duplica as linhas para separar os eventos individualmente
df = df.explode('Tipo de Evento')

# 3. Removemos os espaços que sobram nas pontas após o split (ex: " Evento B" vira "Evento B")
df['Tipo de Evento'] = df['Tipo de Evento'].str.strip()

# 4. Removemos possíveis valores vazios ou nulos que tenham sobrado após a quebra
df['Tipo de Evento'] = df['Tipo de Evento'].replace(['', 'None'], None)
df = df.dropna(subset=['Tipo de Evento'])

# Tratando colunas medicamentos e doenças

Muitas linhas nessas colunas vem com valor 0 ou 'Não' para indicar que a pessoa não tinha doenças ou usava medicamentos.
Devemos padronizar uma forma comum de sinalizar o que é "Sem informação".

In [12]:
colunas_finais = [
    'Doenças (CID10) - Preexistente', 
    'Medicamento em uso anterior ou durante a vacinação', 
    'Nome do Medicamento'
]

for col in colunas_finais:
    # Garantir que tudo seja texto e remover espaços extras nas pontas
    df[col] = df[col].astype(str).str.strip()
    
    # Remover prefixos numéricos 
    df[col] = df[col].str.replace(r'^\d+:\s*', '', regex=True)
    
    # Padronizar respostas negativas ou vazias para facilitar filtros futuros.
    
    valores_vazios = ['0', 'Não', 'Não ', 'nan', 'NaN', '-', '']
    df[col] = df[col].replace(valores_vazios, 'Não se aplica / Nenhum')

# Remoção de colunas duplicadas(geradas durante o tratamento)

In [13]:
colunas_para_remover = [
    'Data da Notificação',       # Já criamos 'Ano_Notificacao' e 'Data_Completa_Notificacao'
    'Imunobiológico (vacina)',   # Já virou 'Vacina_0', 'Vacina_1', etc.
    'Dose'                       # Já virou 'Dose_0', 'Dose_1', etc.
]

# O método .drop remove as colunas. 
# O errors='ignore' serve para o código não quebrar caso você rode o script de novo e a coluna já não exista.
df = df.drop(columns=colunas_para_remover, errors='ignore')

# Salvando localmente

Trecho simples para salvamendo local do novo DataFrame em um xlsx.

In [14]:
nome_do_arquivo = '../data/dados_tratados.xlsx'

df.to_excel(nome_do_arquivo, index=False)

# Gerando gráficos

## Gráfico de barras de tipo de evento X sexo

In [17]:
import plotly.express as px # importando biblioteca para visualização de dados

In [ ]:
# 1. Filtro de Segurança: Vamos criar uma cópia leve tirando o que for nulo ou 'None'
# Isso impede que textos perdidos quebrem o seu gráfico
df_grafico = df[df['Idade Evento'].notna()].copy()
df_grafico = df_grafico[df_grafico['Tipo de Evento'].notna()]

# Garante que a idade seja lida estritamente como número
df_grafico['Idade Evento'] = pd.to_numeric(df_grafico['Idade Evento'], errors='coerce')
df_grafico = df_grafico.dropna(subset=['Idade Evento'])

# 2. Criamos o Histograma (Super limpo e legível)
fig_distribuicao = px.histogram(
    df_grafico, 
    x='Idade Evento', 
    color='Tipo de Evento', 
    nbins=15,                    # Divide as idades em 15 blocos fáceis de ler
    barmode='overlay',           # Sobrepõe as cores com transparência para comparar
    title='Quem são os afetados? Distribuição de Idade por Tipo de Evento',
    labels={'Idade Evento': 'Idade da Pessoa', 'count': 'Quantidade de Casos'},
    color_discrete_sequence=['#4A90E2', '#E2849A']
)

# 3. Ajustes visuais agressivos para garantir legibilidade no VS Code
fig_distribuicao.update_layout(
    plot_bgcolor='white',        # Fundo branco limpo
    font=dict(size=13),          # Aumenta a letra para você conseguir ler sem esforço
    bargap=0.05,                 # Espaço elegante entre as barras
    showlegend=False,             # Esconde a legenda para não poluir o visual
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0) # Legenda deitada no topo
)

# Adiciona linhas de grade horizontais bem discretas
fig_distribuicao.update_yaxes(showgrid=True, gridcolor='LightGrey')

fig_distribuicao.show()